# Disease-resolved arterial spectral resonance: full computational study

This notebook is the **thin orchestration and plotting interface** for the `arterial_spectral_cascade` Python package. The package is the authoritative numerical implementation; this notebook documents the study, selects the run mode, mounts persistent storage in Google Drive, executes the verified calculations, and regenerates publication figures.

The default mode is `FULL_STUDY`.

The notebook does not redefine the governing equations, spectral operators, ETDRK4 scheme, verification tests, or persistence logic. Those are imported from the installed package.

## 1. Governing model and Stage-1 disease parameterization

The solved reduced-order equation is

$$
a_s+a a_\xi+b(\xi)a_{\xi\xi\xi}+g(\xi)\Lambda a=0,
\qquad
\Lambda=(-\partial_\xi^2)^{1/2}.
$$

The anatomical input is a dimensionless radius field $r(\xi)$ defined through

$$
R(x)=R_0r(\xi),
\qquad
\xi=x/L_0,
$$

and the quasi-local Womersley field is

$$
\mathrm{Wo}_R(\xi)=\mathrm{Wo}_0r(\xi).
$$

The Stage-1 coefficient closure is

$$
b(\xi)=b_0(\mathrm{Wo}_0)r(\xi)^{-2}B_G(\xi),
$$

$$
g(\xi)=g_{\mathrm{ref}}
\left(1+\frac{C_g}{\mathrm{Wo}_0r(\xi)}\right)G_G(\xi).
$$

For the disease-only calculations used in the principal comparisons, $\varepsilon_b=\varepsilon_g=0$. Thus disease enters the solver through $r(\xi)$ and the derived fields $\mathrm{Wo}_R(\xi)$, $b(\xi)$, and $g(\xi)$; arbitrary lesion-specific coefficient fields are not introduced.

## 2. Stage-2 numerical method

The coefficient fields are decomposed as

$$
b=\bar b+\widetilde b,
\qquad
g=\bar g+\widetilde g.
$$

The diagonal mean operator

$$
L_0(k)=-i\bar b k^3-\bar g|k|
$$

is advanced analytically by ETDRK4. The nonlinear and heterogeneous residual is

$$
\mathcal F(a)
=-\frac12\partial_\xi(a^2)
-\widetilde b\,a_{\xi\xi\xi}
-\widetilde g\,\Lambda a.
$$

The implementation uses the Stage-2 Fourier convention, symmetric two-thirds de-aliasing, cancellation-safe $\varphi_j$ functions, exact preservation of the dynamically generated zero mode, and the Stage-1 integral-balance identities as numerical diagnostics.

## 3. Evidence-referenced disease representations

The study uses fixed amplitude representations derived from established published studies. The labels are **study identifiers**, not universal clinical-risk grades.

- `S10`, `S20`, `S30`: smooth distributed narrowing with 10%, 20%, and 30% diameter reduction.
- `D20`: smooth distributed dilation with $D_{\max}/D_0=1.20$.
- `D50`: smooth distributed dilation with $D_{\max}/D_0=1.50$.
- `D60`: smooth distributed dilation with $D_{\max}/D_0=1.60$.

The source registry is embedded in the package and persisted with the study tables. Published studies provide the severity-amplitude context; the Stage-1 smooth periodic radius representation remains the model solved here.

## 4. Install the local package

The archive should be extracted so that `arterial_spectral_cascade_package` is present either in the current working directory or at `/content/arterial_spectral_cascade_package` in Google Colab. This cell installs that local source tree in editable mode.

In [ ]:
from pathlib import Path
import os, sys, subprocess

candidates = [
    Path.cwd(),
    Path.cwd() / "arterial_spectral_cascade_package",
    Path("/content/arterial_spectral_cascade_package"),
]
PACKAGE_ROOT = next((p.resolve() for p in candidates if (p / "pyproject.toml").exists()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError(
        "Could not locate arterial_spectral_cascade_package. Extract the supplied zip into "
        "the current directory or /content, then rerun this cell."
    )

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", "-e", str(PACKAGE_ROOT)])
print(f"Installed local package from: {PACKAGE_ROOT}")

## 5. Import the scientific interface and inspect the evidence table

`FULL_STUDY` is the default mode. The available user-facing modes are `QUICK_CHECK`, `VERIFICATION`, `PARAMETER_SELECTION`, `FULL_STUDY`, and `FIGURES`.

In [ ]:
from copy import deepcopy
from arterial_spectral_cascade import __version__
from arterial_spectral_cascade.config import default_study_config
from arterial_spectral_cascade.study import evidence_profile_table, configured_root, run_study_mode
from arterial_spectral_cascade.storage import init_project_paths
from arterial_spectral_cascade.plotting import regenerate_available_figures

STUDY_CONFIG = default_study_config()
STUDY_CONFIG["RUN_MODE"] = "FULL_STUDY"

print("arterial-spectral-cascade version:", __version__)
print("Run mode:", STUDY_CONFIG["RUN_MODE"])
evidence_profile_table()

## 6. Persistent Google Drive storage

In Colab, the default study directory is `/content/drive/MyDrive/PoF_ArterialSpectralCascade`. All full-resolution trajectories are restartable. Case metadata, checkpoints, verification reports, result archives, tables, figures, and logs are written in separate directories.

To use a different location, set `STUDY_CONFIG["PROJECT_ROOT"]` before running the next cell.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and STUDY_CONFIG.get("MOUNT_DRIVE", True):
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_ROOT = configured_root(STUDY_CONFIG)
PATHS = init_project_paths(PROJECT_ROOT)
print("Persistent study directory:", PATHS.root)

## 7. Numerical settings and acceptance logic

The default study starts from $N=512$ and $\Delta s=2\times10^{-4}$, but the `PARAMETER_SELECTION` stage determines the coarsest verified spatial resolution and largest verified fixed timestep common to the highest evidence-referenced grade in each disease branch.

The parent-reference procedure separates two questions:

1. **Numerical acceptance:** the Stage-2 parent trajectories must complete, remain finite, satisfy the constant-coefficient dissipative balance, and pass the independent Stage-2 verification suite.
2. **Legacy topology comparison:** whether the new Stage-2 baseline reproduces the previously reported discrete peak at $\mathrm{Wo}=15$ is recorded as a diagnostic only and is not an acceptance criterion.

The disease calculations therefore measure heterogeneity effects relative to the verified Stage-2 baseline rather than forcing a prescribed resonance topology.

## 8. Locked result set

The full study generates the five result blocks required for the paper:

**R1 — Geometry to heterogeneous spectral coupling.** The solver records $r(\xi)$, $\mathrm{Wo}_R(\xi)$, $b(\xi)$, $g(\xi)$, coefficient spectra, and the heterogeneous coupling matrix $H_{\ell n}$.

**R2 — Disease-dependent resonance landscapes.** Stenosis and dilation are resolved across Womersley number and evidence-referenced severity, with adaptive refinement that does not assume a single interior resonance.

**R3 — Heterogeneous versus matched-mean response.** Every principal disease calculation has a matched-mean control, with

$$
\Delta R(s)=R_{\mathrm{het}}(s)-R_{\mathrm{mm}}(s)
$$

and

$$
D_2(s)=
\frac{\|a_{\mathrm{het}}-a_{\mathrm{mm}}\|_{L^2}}
{\max(\|a_{\mathrm{mm}}\|_{L^2},\epsilon_{\mathrm{mach}})}.
$$

**R4 — Mechanistic modal-energy analysis.** Selected cases resolve

$$
\frac{dE_k}{ds}
=T_N(k)+T_{\widetilde b}(k)+T_{\widetilde g}(k)+T_{\bar g}(k),
$$

with the mean dispersive contribution verified to be phase-only.

**R5 — Axial-scale selectivity.** The selected disease branches are evaluated across admissible lesion scale $w$ to determine whether heterogeneous coupling is scale selective.

## 9. Execute the full study

This is the only computational cell required for a normal complete run. The sequence is interruption-safe and restartable:

`QUICK_CHECK` $\rightarrow$ `VERIFICATION` $\rightarrow$ `PARAMETER_SELECTION` $\rightarrow$ R1–R3 $\rightarrow$ R4 $\rightarrow$ R5 $\rightarrow$ publication figures.

A compatible completed step is reused from persistent storage. Failed or incompatible verification records are not silently accepted.

In [ ]:
STUDY_REPORT = run_study_mode(PATHS, STUDY_CONFIG, progress=True)
STUDY_REPORT

## 10. Physics of Fluids publication figures

The plotting layer uses the locked AIP/*Physics of Fluids* template. Figures are prepared at final publication dimensions, use at least 8-pt text and 0.5-pt lines, distinguish curves by line style and/or marker as well as color, omit internal panel titles, and label multipart panels `(a)`, `(b)`, and so forth.

Each figure is written as vector PDF, vector SVG, 600-dpi PNG, and an alt-text sidecar. Figures are regenerated only from persisted validated result archives.

In [ ]:
FIGURE_FILES = regenerate_available_figures(PATHS, STUDY_CONFIG)
print(f"Publication figure files: {len(FIGURE_FILES)}")
for f in FIGURE_FILES:
    print(f)

## 11. Scope

The calculations establish how disease-resolved radius heterogeneity modifies spectral coupling and resonance within the Stage-1/Stage-2 reduced model. They do not predict plaque progression, rupture, thrombosis, wall stress, wall shear stress, treatment efficacy, or patient outcome. Any medical interpretation must remain within that reduced-order scope.